# P1b — Dialogue builder (R4, R5, R7) + prep_report

Applies R4 (template stripping + credential filter), R5 (English filter, code fences,
first-exchange length gate) and R7 (conservation check).  Outputs:
- `sft/work/dialogues_all.jsonl`  — canonical D2 records
- `sft/reports/prep_report.json` — drop counts per reason × source (R7 closure)

**Privacy rule:** no dataset text in any cell output — counts, ids and hashes only.

## 0. Parameters

In [1]:
import os
from pathlib import Path

# ── repo layout ────────────────────────────────────────────────────────────────
_p = Path().resolve()
REPO_ROOT = _p
for _ in range(6):
    if (_p / "sft").is_dir():
        REPO_ROOT = _p
        break
    _p = _p.parent

SFT_DIR         = REPO_ROOT / "sft"
CSV_PATH        = REPO_ROOT / "data" / "processed" / "integrated_conversations_ner_validated.csv"
SNAP_DIR        = SFT_DIR / "tokenizer_snapshot"
REPORTS_DIR     = SFT_DIR / "reports"
WORK_DIR        = SFT_DIR / "work"
OUTPUT_JSONL    = WORK_DIR / "dialogues_all.jsonl"
PREP_REPORT     = REPORTS_DIR / "prep_report.json"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

# ── E1-derived thresholds (from e1_profile.json D5 and ARCHITECTURE D3 R4) ──
MIN_USER_TOKENS      = 5    # minimum supervised user-turn tokens after strip
MIN_ASST_TOKENS      = 20   # minimum supervised assistant-turn tokens after strip
MAX_LEN              = 2048 # sequence length cap (R5, E3)
MAX_CREDENTIAL_DROP_PCT = 25.0  # R4: if >25% of MedDialog dropped, stop

# ── R4 template patterns (case-insensitive, from E1 D3) ──────────────────────
TEMPLATE_PATTERNS = [
    r"hope i have answered",
    r"chat\s*doctor",
    r"thanks for your query",
    r"dear friend",
    r"hi i am",
    r"hello i am",
]

# R4 credential claim pattern → drop the whole dialogue
CREDENTIAL_PATTERN = r"(?:i am dr\b|as your doctor|i['\'']m a doctor|i am a doctor)"

print(f"REPO_ROOT        : {REPO_ROOT}")
print(f"CSV_PATH exists  : {CSV_PATH.exists()}")
print(f"SNAP_DIR exists  : {SNAP_DIR.exists()}")
print(f"OUTPUT_JSONL     : {OUTPUT_JSONL}")
print(f"PREP_REPORT      : {PREP_REPORT}")
print(f"MIN_USER_TOKENS  : {MIN_USER_TOKENS}")
print(f"MIN_ASST_TOKENS  : {MIN_ASST_TOKENS}")
print(f"MAX_LEN          : {MAX_LEN}")


REPO_ROOT        : C:\Users\nirmi\Desktop\Capstone
CSV_PATH exists  : True
SNAP_DIR exists  : True
OUTPUT_JSONL     : C:\Users\nirmi\Desktop\Capstone\sft\work\dialogues_all.jsonl
PREP_REPORT      : C:\Users\nirmi\Desktop\Capstone\sft\reports\prep_report.json
MIN_USER_TOKENS  : 5
MIN_ASST_TOKENS  : 20
MAX_LEN          : 2048


## 1. Definitions

In [2]:
import gc
import hashlib
import json
import re
import time
import unicodedata
from typing import Optional

import pandas as pd

# ── timing ────────────────────────────────────────────────────────────────────
_TIMINGS: dict = {}

def timer(name: str):
    import contextlib
    @contextlib.contextmanager
    def _ctx():
        t0 = time.perf_counter()
        yield
        _TIMINGS[name] = round(time.perf_counter() - t0, 2)
    return _ctx()

# ── R6 normalizer ────────────────────────────────────────────────────────────
_BLANK_RUN = re.compile(r"\n{3,}")

def normalize_text(text: str) -> str:
    """R6: Unicode NFC, strip, collapse blank-line runs (≥3 newlines → 2)."""
    text = unicodedata.normalize("NFC", text)
    text = text.strip()
    text = _BLANK_RUN.sub("\n\n", text)
    return text

# ── compiled patterns ─────────────────────────────────────────────────────────
_TEMPLATE_RE   = re.compile("|".join(TEMPLATE_PATTERNS), re.IGNORECASE)
_CREDENTIAL_RE = re.compile(CREDENTIAL_PATTERN, re.IGNORECASE)
_CODE_FENCE_RE = re.compile(r"```|\|\s*[-:]+\s*\|")   # code fences or markdown tables

def strip_template(text: str) -> str:
    """
    R4: Remove template sentences (sign-offs, greetings) from assistant text.
    Splits on sentence boundaries ('.', '!', '?', '\n'), removes matching sentences,
    re-joins and normalizes.  Does NOT drop sentences that are the only content.
    Returns the stripped text (may be empty if all sentences matched).
    """
    # Split into sentences: keep delimiter by using re.split with capture
    parts = re.split(r'(?<=[.!?])\s+|\n', text)
    kept = []
    for part in parts:
        if part.strip() and not _TEMPLATE_RE.search(part):
            kept.append(part.strip())
    return " ".join(kept).strip()

def has_credential_claim(text: str) -> bool:
    """R4: True if text contains a first-person credential claim."""
    return bool(_CREDENTIAL_RE.search(text))

def has_code_fence(text: str) -> bool:
    """R5: True if text contains code fences or markdown tables."""
    return bool(_CODE_FENCE_RE.search(text))

# ── tokenizer (for length gating) ─────────────────────────────────────────────
os.environ["HF_HUB_OFFLINE"] = "1"
from transformers import AutoTokenizer
_tok = AutoTokenizer.from_pretrained(str(SNAP_DIR), local_files_only=True)

def token_count(text: str) -> int:
    return len(_tok.encode(text, add_special_tokens=False))

# ── ChatML renderer (section D4, no <think>) ─────────────────────────────────
_SYSTEM_PROMPT = (SFT_DIR / "config" / "system_prompt_v1.txt").read_text(encoding="utf-8")

_IM_START = "<|im_start|>"
_IM_END   = "<|im_end|>"

def render_chatml(messages: list, include_gen_prompt: bool = False) -> str:
    """
    Render a list of {role, content} messages in our ChatML format.
    messages must already be normalized (R6).
    """
    parts = [
        f"{_IM_START}system\n{_SYSTEM_PROMPT}{_IM_END}\n"
    ]
    for msg in messages:
        role    = msg["role"]
        content = msg["content"]
        parts.append(f"{_IM_START}{role}\n{content}{_IM_END}\n")
    if include_gen_prompt:
        parts.append(f"{_IM_START}assistant\n")
    return "".join(parts)

def sequence_token_count(messages: list) -> int:
    """Token count of a full ChatML sequence (system + all messages)."""
    return token_count(render_chatml(messages))

# ── R1-R3 cut-at-first-defect (same as P1a) ──────────────────────────────────
def cut_at_first_defect(turns: list) -> tuple:
    """
    R1-R3 on a sorted list of {turn_id, speaker, utterance} dicts.
    Returns (valid_prefix: list, defect_reason: str|None).
    """
    valid   = []
    defect  = None
    for i, t in enumerate(turns):
        tid  = t["turn_id"]
        role = t["speaker"]   # 'user' or 'assistant'
        utt  = t["utterance"]
        # R2 contiguity
        if tid != i:
            defect = "non_contiguous_turn_id"
            break
        # R2 role alternation starting with user
        expected_role = "user" if i % 2 == 0 else "assistant"
        if role != expected_role:
            defect = "wrong_role_order"
            break
        # R2 non-null, non-empty
        if utt is None or (isinstance(utt, float) and utt != utt):
            defect = "null_or_empty_utterance"
            break
        text = normalize_text(str(utt))
        if not text:
            defect = "null_or_empty_utterance"
            break
        valid.append({"turn_id": tid, "speaker": role, "content": text})
    # R3: keep longest prefix ending on assistant
    if defect is not None:
        # trim to last assistant turn
        while valid and valid[-1]["speaker"] != "assistant":
            valid.pop()
    return valid, defect

def count_exchanges(turns: list) -> int:
    return sum(1 for t in turns if t.get("speaker") == "assistant")

print("Section 1 definitions loaded.")
print(f"System prompt SHA-256 : {hashlib.sha256(_SYSTEM_PROMPT.encode('utf-8')).hexdigest()}")
print(f"System prompt length  : {len(_SYSTEM_PROMPT)} chars")


C:\Users\nirmi\Desktop\Capstone\sft\.venv-nb\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Section 1 definitions loaded.
System prompt SHA-256 : e68af7244b7f6332620cd69a67cd4c17e800e5b59401067765cb1903ffa5b0a4
System prompt length  : 508 chars


## 2. In-notebook tests (synthetic fixtures — must pass before any real data)

In [3]:
import unittest

class TestR4(unittest.TestCase):

    def test_strip_template_removes_signoff(self):
        text = "Your headache may be tension-type. Hope I have answered your query. Take care."
        result = strip_template(text)
        self.assertNotIn("Hope I have answered", result)
        self.assertIn("tension-type", result)

    def test_strip_template_removes_chatdoctor(self):
        text = "Take two cups of water daily. Chat Doctor recommends this. Rest well."
        result = strip_template(text)
        self.assertNotIn("Chat Doctor", result)
        self.assertIn("Take two cups", result)

    def test_strip_template_noop_on_clean(self):
        text = "Please see a doctor. Your symptoms suggest a check-up is needed."
        result = strip_template(text)
        self.assertEqual(result, text)

    def test_credential_flag(self):
        self.assertTrue(has_credential_claim("I am Dr Smith and will help you."))
        self.assertTrue(has_credential_claim("as your doctor, I recommend rest."))
        self.assertFalse(has_credential_claim("I recommend seeing a doctor."))

    def test_credential_no_false_positive_on_dr_title_elsewhere(self):
        # 'Dr' in a non-first-person context should not trigger
        self.assertFalse(has_credential_claim("Dr Smith will call you."))

class TestR5(unittest.TestCase):

    def test_code_fence_detected(self):
        self.assertTrue(has_code_fence("Here is a script:\n```python\nprint()\n```"))

    def test_markdown_table_detected(self):
        self.assertTrue(has_code_fence("| Col1 | Col2 |\n|---|---|"))

    def test_clean_text_no_fence(self):
        self.assertFalse(has_code_fence("Your blood pressure reading is normal."))

class TestCutAtFirstDefect(unittest.TestCase):

    def _turn(self, tid, role, utt):
        return {"turn_id": tid, "speaker": role, "utterance": utt}

    def test_valid_two_turn(self):
        turns = [self._turn(0, "user", "My head hurts."),
                 self._turn(1, "assistant", "Please see a clinician.")]
        valid, defect = cut_at_first_defect(turns)
        self.assertIsNone(defect)
        self.assertEqual(len(valid), 2)

    def test_non_contiguous_drops_to_last_assistant(self):
        turns = [self._turn(0, "user", "Hello"),
                 self._turn(1, "assistant", "Hi there"),
                 self._turn(3, "user", "skipped")]   # gap: turn_id 3 not 2
        valid, defect = cut_at_first_defect(turns)
        self.assertEqual(defect, "non_contiguous_turn_id")
        self.assertEqual(count_exchanges(valid), 1)

    def test_null_utterance_cut(self):
        turns = [self._turn(0, "user", "Hello"),
                 self._turn(1, "assistant", None)]
        valid, defect = cut_at_first_defect(turns)
        self.assertEqual(defect, "null_or_empty_utterance")
        self.assertEqual(len(valid), 0)  # no full exchange remains

    def test_wrong_role_order(self):
        turns = [self._turn(0, "user", "Hello"),
                 self._turn(1, "user", "Also")]  # user at turn 1, should be assistant
        valid, defect = cut_at_first_defect(turns)
        self.assertEqual(defect, "wrong_role_order")
        self.assertEqual(len(valid), 0)

class TestRenderChatML(unittest.TestCase):

    def test_starts_with_system(self):
        msgs = [{"role": "user", "content": "Hello"}, {"role": "assistant", "content": "Hi"}]
        rendered = render_chatml(msgs)
        self.assertTrue(rendered.startswith("<|im_start|>system\n"))

    def test_ends_with_im_end_newline(self):
        msgs = [{"role": "user", "content": "Test"}, {"role": "assistant", "content": "Ok"}]
        rendered = render_chatml(msgs)
        self.assertTrue(rendered.endswith("<|im_end|>\n"))

    def test_gen_prompt_appended(self):
        msgs = [{"role": "user", "content": "Hi"}]
        rendered = render_chatml(msgs, include_gen_prompt=True)
        self.assertTrue(rendered.endswith("<|im_start|>assistant\n"))

    def test_no_think_blocks(self):
        msgs = [{"role": "user", "content": "What is two plus two?"}, {"role": "assistant", "content": "Four."}]
        rendered = render_chatml(msgs)
        self.assertNotIn("<think>", rendered)

# ── Run tests ─────────────────────────────────────────────────────────────────
loader  = unittest.TestLoader()
suite   = unittest.TestSuite()
for cls in [TestR4, TestR5, TestCutAtFirstDefect, TestRenderChatML]:
    suite.addTests(loader.loadTestsFromTestCase(cls))

runner = unittest.TextTestRunner(verbosity=2, stream=__import__("sys").stdout)
result = runner.run(suite)
assert result.wasSuccessful(), f"In-notebook tests FAILED: {result.failures} {result.errors}"
print(f"\nAll {result.testsRun} in-notebook tests PASSED.")


test_credential_flag (__main__.TestR4.test_credential_flag) ... 

ok


test_credential_no_false_positive_on_dr_title_elsewhere (__main__.TestR4.test_credential_no_false_positive_on_dr_title_elsewhere) ... 

ok


test_strip_template_noop_on_clean (__main__.TestR4.test_strip_template_noop_on_clean) ... 

ok


... (24 outputs left out)

test_starts_with_system (__main__.TestRenderChatML.test_starts_with_system) ... 

ok


----------------------------------------------------------------------
Ran 16 tests in 0.007s

OK



All 16 in-notebook tests PASSED.


## 3. Streaming dialogue builder (R1–R7)

In [4]:
import tracemalloc
tracemalloc.start()

DTYPES = {
    "dialogue_id":    "str",
    "turn_id":        "Int64",
    "speaker":        "str",
    "utterance":      "object",
    "source_dataset": "str",
    "original_id":    "str",
}
COLS = list(DTYPES.keys())

# ── drop-reason counters (per source) ─────────────────────────────────────────
SOURCES = ["MedDialog", "HealthChat-LMSYS", "HealthChat-WildChat"]
DROP_REASONS = [
    "r3_defect_no_exchange",    # R3: cut left no full exchange
    "r4_credential",            # R4: first-person credential claim
    "r4_strip_too_short",       # R4: assistant too short after stripping
    "r5_not_english",           # R5: langid detected non-English user turn
    "r5_code_fence",            # R5: code fence or markdown table
    "r5_first_exchange_too_long", # R5: first exchange exceeds MAX_LEN alone
]

# counts[source][reason] = int
counts = {src: {"kept": 0, **{r: 0 for r in DROP_REASONS}} for src in SOURCES}

# ── lazy langid import ────────────────────────────────────────────────────────
import langid
langid.set_languages(["en", "de", "fr", "es", "nl", "it", "no", "sv", "da", "pt"])  # prune

def is_english(text: str) -> bool:
    lang, _ = langid.classify(text)
    return lang == "en"

# ── output writer ─────────────────────────────────────────────────────────────
with timer("build"):
    reader = pd.read_csv(
        CSV_PATH,
        usecols=COLS,
        dtype=DTYPES,
        chunksize=50_000,
        encoding="utf-8",
        low_memory=False,
    )

    # Accumulate full dialogues across chunks via buffer
    _buf: dict = {}  # dialogue_id -> list of rows
    written = 0

    out_fh = open(OUTPUT_JSONL, "w", encoding="utf-8", newline="\n")

    def flush_dialogue(did: str, rows: list) -> None:
        """Apply R1-R7 to a dialogue's rows and write if kept."""
        global written
        src = rows[0]["source_dataset"]
        if src not in counts:
            return  # unexpected source — skip silently

        original_id = rows[0]["original_id"]

        # Sort by turn_id (R1)
        rows_sorted = sorted(rows, key=lambda r: int(r["turn_id"]) if r["turn_id"] == r["turn_id"] else -1)

        # R1-R3 cut
        raw_turns = [
            {"turn_id": int(r["turn_id"]) if r["turn_id"] == r["turn_id"] else -1,
             "speaker":   str(r["speaker"]),
             "utterance": r["utterance"]}
            for r in rows_sorted
        ]
        valid, _ = cut_at_first_defect(raw_turns)
        if count_exchanges(valid) == 0:
            counts[src]["r3_defect_no_exchange"] += 1
            return

        # R4 credential check (before stripping — check raw content)
        for t in valid:
            if t["speaker"] == "assistant" and has_credential_claim(t["content"]):
                counts[src]["r4_credential"] += 1
                return

        # R4 template stripping (inline, never deletes turns)
        templates_stripped = False
        for t in valid:
            if t["speaker"] == "assistant":
                stripped = strip_template(t["content"])
                if stripped != t["content"]:
                    templates_stripped = True
                    t["content"] = stripped

        # R4 min-length after stripping
        for t in valid:
            role = t["speaker"]
            min_tok = MIN_ASST_TOKENS if role == "assistant" else MIN_USER_TOKENS
            if token_count(t["content"]) < min_tok:
                counts[src]["r4_strip_too_short"] += 1
                return

        # R5 English filter on user turns
        user_turns = [t["content"] for t in valid if t["speaker"] == "user"]
        # check first user turn (most reliable)
        if user_turns and not is_english(user_turns[0]):
            counts[src]["r5_not_english"] += 1
            return

        # R5 code fence / markdown table in any turn
        for t in valid:
            if has_code_fence(t["content"]):
                counts[src]["r5_code_fence"] += 1
                return

        # R5 first-exchange length gate: first exchange must fit MAX_LEN
        first_two = valid[:2]
        msgs_first = [{"role": t["speaker"], "content": t["content"]} for t in first_two]
        if sequence_token_count(msgs_first) > MAX_LEN:
            counts[src]["r5_first_exchange_too_long"] += 1
            return

        # E3 truncation: drop trailing exchanges until sequence fits MAX_LEN
        # Keep longest prefix ending on assistant turn that fits
        truncated_exchanges = 0
        msgs_all = [{"role": t["speaker"], "content": t["content"]} for t in valid]
        while len(msgs_all) > 2 and sequence_token_count(msgs_all) > MAX_LEN:
            # drop last two (user+assistant)
            msgs_all = msgs_all[:-2]
            truncated_exchanges += 1

        # Re-compute valid messages after truncation
        n_exchanges = sum(1 for m in msgs_all if m["role"] == "assistant")

        # Build D2 record
        messages_out = [
            {"turn_id": i, "role": m["role"], "content": m["content"]}
            for i, m in enumerate(msgs_all)
        ]
        record = {
            "dialogue_id": did,
            "source":      src,
            "original_id": original_id,
            "group_id":    None,   # assigned in P2a
            "split":       None,   # assigned in P2a
            "n_exchanges": n_exchanges,
            "messages":    messages_out,
            "meta": {
                "templates_stripped":   templates_stripped,
                "truncated_exchanges":  truncated_exchanges,
                "n_tokens":             sequence_token_count(msgs_all),
            },
        }
        out_fh.write(json.dumps(record, sort_keys=True, ensure_ascii=False) + "\n")
        counts[src]["kept"] += 1
        written += 1
        if written % 10_000 == 0:
            _, peak = tracemalloc.get_traced_memory()
            print(f"  {written:>7,} written  peak={peak/1_048_576:.1f} MB", flush=True)

    # Stream chunks
    current_did = None
    for chunk in reader:
        for _, row in chunk.iterrows():
            did_row = str(row["dialogue_id"])
            if did_row != current_did:
                if current_did is not None and current_did in _buf:
                    flush_dialogue(current_did, _buf.pop(current_did))
                current_did = did_row
            _buf.setdefault(current_did, []).append(dict(row))
    # Flush last dialogue
    if current_did is not None and current_did in _buf:
        flush_dialogue(current_did, _buf.pop(current_did))

    out_fh.flush()
    out_fh.close()

_, peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
peak_mb = peak_bytes / 1_048_576

print(f"Build complete: {written:,} dialogues written")
print(f"Peak traced memory: {peak_mb:.1f} MB")
print(f"Build time: {_TIMINGS.get('build','?')}s")


   10,000 written  peak=125.6 MB


   20,000 written  peak=125.6 MB


   30,000 written  peak=125.6 MB


   40,000 written  peak=125.6 MB


   50,000 written  peak=125.6 MB


   60,000 written  peak=125.6 MB


... (2 outputs left out)

   90,000 written  peak=125.6 MB


  100,000 written  peak=125.6 MB


  110,000 written  peak=125.6 MB


Build complete: 119,067 dialogues written
Peak traced memory: 125.6 MB
Build time: 1898.19s


## 4. R7 Conservation check

In [5]:
# Per ARCHITECTURE D3 R7: input = kept + sum(dropped-by-reason) per source
# Source totals from G1 (verified in P1a)
G1_DIALOGUES = {
    "MedDialog":           112_164,
    "HealthChat-LMSYS":      6_828,
    "HealthChat-WildChat":   4_264,
}

r7_ok = True
r7_rows = []
for src in SOURCES:
    c       = counts[src]
    kept    = c["kept"]
    dropped = sum(c[r] for r in DROP_REASONS)
    total   = kept + dropped
    expected = G1_DIALOGUES.get(src, 0)
    match   = (total == expected)
    if not match:
        r7_ok = False
    r7_rows.append({
        "source": src,
        "kept": kept,
        "dropped": dropped,
        "total": total,
        "expected": expected,
        "match": match,
    })

print("R7 Conservation check:")
print(f"  {'source':<28}  {'kept':>8}  {'dropped':>8}  {'total':>8}  {'expected':>8}  {'match'}")
for r in r7_rows:
    print(f"  {r['source']:<28}  {r['kept']:>8,}  {r['dropped']:>8,}  {r['total']:>8,}  {r['expected']:>8,}  {'OK' if r['match'] else 'MISMATCH'}")

assert r7_ok, "R7 conservation FAILED — counts do not close. Check drop reasons."
print("\nR7 conservation CLOSED.")

# MedDialog credential drop threshold check (R4: must not drop >25% of MedDialog)
md_counts = counts["MedDialog"]
md_cred_pct = 100.0 * md_counts["r4_credential"] / G1_DIALOGUES["MedDialog"]
print(f"\nMedDialog credential drop: {md_counts['r4_credential']:,} ({md_cred_pct:.2f}%)")
assert md_cred_pct <= MAX_CREDENTIAL_DROP_PCT, (
    f"R4 credential filter drops {md_cred_pct:.2f}% of MedDialog (>{MAX_CREDENTIAL_DROP_PCT}%): "
    "stop and review the regex."
)
print("R4 credential filter within threshold.")


R7 Conservation check:
  source                            kept   dropped     total  expected  match
  MedDialog                      109,577     2,587   112,164   112,164  OK
  HealthChat-LMSYS                 5,924       904     6,828     6,828  OK
  HealthChat-WildChat              3,566       698     4,264     4,264  OK

R7 conservation CLOSED.

MedDialog credential drop: 230 (0.21%)
R4 credential filter within threshold.


## 5. Validate written JSONL (schema + alternation + residual template check)

In [6]:
import hashlib

_REQUIRED_FIELDS = {
    "dialogue_id", "source", "original_id", "group_id", "split",
    "n_exchanges", "messages", "meta"
}
_MSG_FIELDS = {"turn_id", "role", "content"}
_META_FIELDS = {"templates_stripped", "truncated_exchanges", "n_tokens"}

stats = {
    "total": 0,
    "schema_errors": 0,
    "alternation_errors": 0,
    "residual_template_asst_turns": 0,
    "total_asst_turns": 0,
    "sources": {s: 0 for s in SOURCES},
    "meddialog_non_single": 0,
    "multi_exchange": 0,
    "sha256": "",
}

_hasher = hashlib.sha256()

with open(OUTPUT_JSONL, "r", encoding="utf-8") as fh:
    for lineno, line in enumerate(fh, 1):
        _hasher.update(line.encode("utf-8"))
        rec = json.loads(line)
        stats["total"] += 1

        # Schema
        if not _REQUIRED_FIELDS.issubset(rec.keys()):
            stats["schema_errors"] += 1
            continue

        src = rec["source"]
        if src in stats["sources"]:
            stats["sources"][src] += 1

        if rec["n_exchanges"] > 1:
            stats["multi_exchange"] += 1
        if src == "MedDialog" and rec["n_exchanges"] != 1:
            stats["meddialog_non_single"] += 1

        msgs = rec["messages"]
        # Alternation check
        for i, m in enumerate(msgs):
            expected_role = "user" if i % 2 == 0 else "assistant"
            if m["role"] != expected_role or m["turn_id"] != i:
                stats["alternation_errors"] += 1
                break

        # Residual template check
        for m in msgs:
            if m["role"] == "assistant":
                stats["total_asst_turns"] += 1
                if _TEMPLATE_RE.search(m["content"]):
                    stats["residual_template_asst_turns"] += 1

stats["sha256"] = _hasher.hexdigest()

resid_pct = (
    100.0 * stats["residual_template_asst_turns"] / stats["total_asst_turns"]
    if stats["total_asst_turns"] else 0.0
)

print("JSONL validation:")
print(f"  Total records        : {stats['total']:,}")
print(f"  Schema errors        : {stats['schema_errors']}")
print(f"  Alternation errors   : {stats['alternation_errors']}")
print(f"  MedDialog non-single : {stats['meddialog_non_single']}")
print(f"  Multi-exchange       : {stats['multi_exchange']:,}")
print(f"  By source:")
for src, cnt in stats['sources'].items():
    print(f"    {src:<28}: {cnt:,}")
print(f"  Total asst turns     : {stats['total_asst_turns']:,}")
print(f"  Residual template    : {stats['residual_template_asst_turns']:,} ({resid_pct:.4f}%)")
print(f"  SHA-256              : {stats['sha256']}")

# Acceptance criteria
assert stats["schema_errors"] == 0, f"{stats['schema_errors']} schema errors"
assert stats["alternation_errors"] == 0, f"{stats['alternation_errors']} alternation errors"
assert stats["meddialog_non_single"] == 0, f"{stats['meddialog_non_single']} MedDialog non-single"
assert resid_pct <= 0.1, f"Residual template {resid_pct:.4f}% exceeds 0.1% threshold"
print("\nAll validation assertions PASSED.")


JSONL validation:
  Total records        : 119,067
  Schema errors        : 0
  Alternation errors   : 0
  MedDialog non-single : 0
  Multi-exchange       : 3,576
  By source:
    MedDialog                   : 109,577
    HealthChat-LMSYS            : 5,924
    HealthChat-WildChat         : 3,566
  Total asst turns     : 126,449
  Residual template    : 0 (0.0000%)
  SHA-256              : 8e605c01e14a83905084c3468321a26982775a278a1b8be82e70e9b6d03292f7

All validation assertions PASSED.


## 6. Write prep_report.json

In [7]:
# Derive n_exchanges histogram from JSONL (no re-read of full data)
_exc_hist: dict = {}
with open(OUTPUT_JSONL, "r", encoding="utf-8") as fh:
    for line in fh:
        rec = json.loads(line)
        ne  = rec["n_exchanges"]
        _exc_hist[ne] = _exc_hist.get(ne, 0) + 1

# Count HealthChat multi-exchange (n_exchanges >= 2)
_hc_multi = sum(
    _exc_hist.get(ne, 0)
    for ne in _exc_hist
    if ne >= 2
) - _exc_hist.get(  # subtract MedDialog contribution (all n_exchanges==1)
    1, 0
)   # this is wrong — recalculate from sources

# Better: track per-source multi counts
_src_multi: dict = {s: 0 for s in SOURCES}
_src_single: dict = {s: 0 for s in SOURCES}
with open(OUTPUT_JSONL, "r", encoding="utf-8") as fh:
    for line in fh:
        rec = json.loads(line)
        src = rec["source"]
        if src in _src_multi:
            if rec["n_exchanges"] >= 2:
                _src_multi[src] += 1
            else:
                _src_single[src] += 1

prep_report = {
    "output_path":       str(OUTPUT_JSONL),
    "output_sha256":     stats["sha256"],
    "total_kept":        stats["total"],
    "total_asst_turns":  stats["total_asst_turns"],
    "residual_template_pct": round(resid_pct, 6),
    "n_exchanges_histogram": {str(k): v for k, v in sorted(_exc_hist.items())},
    "per_source": {
        src: {
            "kept":         counts[src]["kept"],
            "multi_exchange": _src_multi[src],
            "single_exchange": _src_single[src],
            "drop_reasons": {r: counts[src][r] for r in DROP_REASONS},
        }
        for src in SOURCES
    },
    "r7_conservation": [
        {"source": r["source"], "kept": r["kept"], "dropped": r["dropped"],
         "total": r["total"], "expected": r["expected"], "match": r["match"]}
        for r in r7_rows
    ],
    "build_time_s": _TIMINGS.get("build"),
    "peak_memory_mb": round(peak_mb, 1),
}

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
with open(PREP_REPORT, "w", encoding="utf-8", newline="\n") as fh:
    json.dump(prep_report, fh, sort_keys=True, ensure_ascii=False, indent=2)

print(f"prep_report written: {PREP_REPORT}")
print(f"  total kept     : {prep_report['total_kept']:,}")
print(f"  residual tpl % : {prep_report['residual_template_pct']}")
for src in SOURCES:
    p = prep_report["per_source"][src]
    print(f"  {src:<28}: kept={p['kept']:,}  multi={p['multi_exchange']:,}  single={p['single_exchange']:,}")


prep_report written: C:\Users\nirmi\Desktop\Capstone\sft\reports\prep_report.json
  total kept     : 119,067
  residual tpl % : 0.0
  MedDialog                   : kept=109,577  multi=0  single=109,577
  HealthChat-LMSYS            : kept=5,924  multi=1,936  single=3,988
  HealthChat-WildChat         : kept=3,566  multi=1,640  single=1,926


## 7. Final status

In [8]:
import sys

print("=" * 60)
print("P1b — Dialogue Builder: COMPLETE")
print("=" * 60)
print(f"Output JSONL  : {OUTPUT_JSONL}")
print(f"  SHA-256     : {stats['sha256']}")
print(f"  Records     : {stats['total']:,}")
print(f"prep_report   : {PREP_REPORT}")
print()
print("Acceptance criteria:")
print(f"  R7 conservation  : {'CLOSED' if r7_ok else 'FAILED'}")
print(f"  Schema errors    : {stats['schema_errors']}  (required 0)")
print(f"  Alternation err  : {stats['alternation_errors']}  (required 0)")
print(f"  MedDialog single : {'100%' if stats['meddialog_non_single']==0 else 'FAIL'}")
print(f"  Residual tpl     : {resid_pct:.4f}%  (required <=0.1%)")
print()
print(f"Python: {sys.version}")

# Final assertions
assert r7_ok,                          "R7 conservation not closed"
assert stats['schema_errors'] == 0,    "Schema errors found"
assert stats['alternation_errors'] == 0, "Alternation errors found"
assert stats['meddialog_non_single'] == 0, "MedDialog has non-single-exchange records"
assert resid_pct <= 0.1,               f"Residual template {resid_pct:.4f}% > 0.1%"
print("\n✓ All acceptance criteria PASSED. P1b complete.")


P1b — Dialogue Builder: COMPLETE
Output JSONL  : C:\Users\nirmi\Desktop\Capstone\sft\work\dialogues_all.jsonl
  SHA-256     : 8e605c01e14a83905084c3468321a26982775a278a1b8be82e70e9b6d03292f7
  Records     : 119,067
prep_report   : C:\Users\nirmi\Desktop\Capstone\sft\reports\prep_report.json

Acceptance criteria:
  R7 conservation  : CLOSED
  Schema errors    : 0  (required 0)
  Alternation err  : 0  (required 0)
  MedDialog single : 100%
  Residual tpl     : 0.0000%  (required <=0.1%)

Python: 3.13.5 (tags/v3.13.5:6cb20a2, Jun 11 2025, 16:15:46) [MSC v.1943 64 bit (AMD64)]

✓ All acceptance criteria PASSED. P1b complete.
